In [1]:
!pip install -q datasets seqeval

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import numpy as np
import pandas as pd
import random
import math
import pickle
from collections import Counter, defaultdict
import os
import urllib
import zipfile
import json
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

from datasets import load_dataset
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score

random.seed(42)
np.random.seed(42)

In [4]:
#Loading the dataset
raw =load_dataset("eriktks/conll2003", revision="refs/convert/parquet")
print(raw)

conll2003/train/0000.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

conll2003/train/0000.parquet: downloading bytes:           |  0.00B            

0000.parquet:   0%|          | 0.00/312k [00:00<?, ?B/s]

0000.parquet:   0%|          | 0.00/283k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})


In [5]:
# Getting the ClassLabel from ner_tags
Labels = raw["train"].features["ner_tags"].feature.names
print("Total No of tags: ",len(Labels))
print(Labels)


Total No of tags:  9
['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


In [6]:
# Loading the ClassLabel
NER_Feature = raw["train"].features["ner_tags"].feature

# Every sentence in data is converted to token and tag pairs
def pairing(split):
    sentences = []
    for sen in raw[split]:
        tokens = list(sen["tokens"])
        tags = []
        for tag in sen["ner_tags"]:
            tags.append(NER_Feature.int2str(tag))
        sentences.append((tokens,tags))

    return sentences


sets = ["train","validation","test"]
data = {split:pairing(split) for split in sets}

In [7]:
train_vocabulary = Counter([w for tok,_ in data["train"] for w in tok])
test_vocabulary = [w for tok,_ in data["test"] for w in tok]
print(f"Training vocabulary size: {len(train_vocabulary)}")

out_of_vocab = sum([w not in train_vocabulary for w in test_vocabulary])
print(f"Total no of out of vocabulary words is:{out_of_vocab}")


Training vocabulary size: 23623
Total no of out of vocabulary words is:5656


In [8]:
Glove_urls = [
    "https://huggingface.co/stanfordnlp/glove/resolve/main/glove.6B.zip",
    "https://nlp.stanford.edu/data/glove.6B.zip",
]

def glove_embeddings(Glove_urls):
    for url in Glove_urls:
        urllib.request.urlretrieve(url, "glove.6B.zip")
        with zipfile.ZipFile("glove.6B.zip") as zf:
            zf.extract("glove.6B.100d.txt",".")
        os.remove("glove.6B.zip")
        return "glove.6B.100d.txt"

def load_glove(path,dim=100):
    vectors = {}
    with open(path, encoding="utf8") as f:
        for line in f:
            parts = line.rstrip().split(" ")
            if len(parts) != dim + 1:
                continue
            vectors[parts[0]] = np.asarray(parts[1:], dtype=np.float32)
    return vectors

glove = load_glove(glove_embeddings(Glove_urls))
print("Length of glove vector: ",len(glove))



Length of glove vector:  400001


In [9]:
PAD, UNK = "<PAD>", "<UNK>"

def norm(word):
    return "".join("0" if ch.isdigit() else ch for ch in word.lower())

train_word_frequency = Counter(norm(w) for tokens,_ in data["train"] for w in tokens)
word_list = [PAD, UNK] + sorted(train_word_frequency)
seen = set(word_list)

for split in ["validation","test"]:
  for tokens,_ in data[split]:
    for w in tokens:
      k = norm(w)
      if k not in seen and k in glove:
        word_list.append(k)
        seen.add(k)

wordToId = {w:i for i,w in enumerate(word_list)}

char_list = [PAD, UNK] + sorted({ch for toks, _ in data["train"] for w in toks for ch in w})
charTOId = {c:i for i,c in enumerate(char_list)}

tagToId = {t:i for i,t in enumerate(Labels)}
singletons = {wordToId[w] for w,c in train_word_frequency.items() if c==1}

print(f"word vocabulary: {len(word_list)}")
print(f"char vocabulary: {len(char_list)}")
print(f"singletons: {len(singletons)}")

word vocabulary: 21215
char vocabulary: 86
singletons: 7602


In [10]:
emb_dim = 100
emb_matrix = np.random.uniform(-np.sqrt(3 / emb_dim), np.sqrt(3 / emb_dim),(len(word_list), emb_dim)).astype(np.float32)

emb_matrix[wordToId[PAD]] = 0.0

found = 0
for w, i in wordToId.items():
    if w in glove:
        emb_matrix[i] = glove[w]
        found += 1
print(f"We got {found} / {len(word_list)} words ({100 * found / len(word_list):.2f}%)")

del glove

We got 19695 / 21215 words (92.84%)


In [11]:
def encoding(tokens,tags):
  w = [wordToId.get(norm(t),wordToId[UNK]) for t in tokens]
  c = [[charTOId.get(ch,charTOId[UNK]) for ch in t] for t in tokens]
  y = [tagToId[t] for t in tags]
  return w,c,y

In [12]:
encoded = {split: [encoding(*s) for s in sents] for split, sents in data.items()}

def make_batch(examples, train=False):
    B = len(examples)
    max_len = max(len(w) for w, _, _ in examples)
    max_chr = max(len(cw) for _, c, _ in examples for cw in c)

    words = torch.zeros(B, max_len, dtype=torch.long)
    chars = torch.zeros(B, max_len, max_chr, dtype=torch.long)
    tags = torch.zeros(B, max_len, dtype=torch.long)
    mask = torch.zeros(B, max_len, dtype=torch.bool)

    for b, (w, c, y) in enumerate(examples):
        if train:
            w = [wordToId[UNK] if (i in singletons and random.random() < 0.5) else i for i in w]
        words[b, :len(w)] = torch.tensor(w)
        tags[b, :len(y)] = torch.tensor(y)
        mask[b, :len(w)] = True
        for j, cw in enumerate(c):
            chars[b, j, :len(cw)] = torch.tensor(cw)
    return words.to(device), chars.to(device), tags.to(device), mask.to(device)


In [13]:
class CRF(nn.Module):
    def __init__(self, num_tags):
        super().__init__()
        self.num_tags = num_tags
        # transitions[i, j] = score of moving from tag i to tag j
        self.transitions = nn.Parameter(torch.empty(num_tags, num_tags).uniform_(-0.1, 0.1))
        self.start = nn.Parameter(torch.empty(num_tags).uniform_(-0.1, 0.1))
        self.end = nn.Parameter(torch.empty(num_tags).uniform_(-0.1, 0.1))

    def path_score(self, emissions, tags, mask):
        # score of the gold path, one number per sentence
        B, L, _ = emissions.shape
        idx = torch.arange(B, device=emissions.device)
        score = self.start[tags[:, 0]] + emissions[idx, 0, tags[:, 0]]
        for i in range(1, L):
            step = self.transitions[tags[:, i - 1], tags[:, i]] + emissions[idx, i, tags[:, i]]
            score = score + step * mask[:, i]
        last = tags[idx, mask.sum(1) - 1]
        return score + self.end[last]

    def log_partition(self, emissions, mask):
        # log of the summed exp-score over ALL tag paths (forward algorithm)
        alpha = self.start + emissions[:, 0]                       # (B, T)
        for i in range(1, emissions.size(1)):
            nxt = torch.logsumexp(alpha.unsqueeze(2) + self.transitions + emissions[:, i].unsqueeze(1), dim=1)
            alpha = torch.where(mask[:, i].unsqueeze(1), nxt, alpha)
        return torch.logsumexp(alpha + self.end, dim=1)

    def neg_log_likelihood(self, emissions, tags, mask):
        return (self.log_partition(emissions, mask) - self.path_score(emissions, tags, mask)).mean()

    def decode(self, emissions, mask):
        # Viterbi, batched
        B, L, T = emissions.shape
        score = self.start + emissions[:, 0]
        history = []
        for i in range(1, L):
            cand = score.unsqueeze(2) + self.transitions          # (B, prev, cur)
            best, best_prev = cand.max(dim=1)
            nxt = best + emissions[:, i]
            score = torch.where(mask[:, i].unsqueeze(1), nxt, score)
            history.append(best_prev)
        score = score + self.end

        # follow the back-pointers on the CPU (much faster than many small GPU reads)
        lengths = mask.sum(1).tolist()
        best_last = score.argmax(dim=1).tolist()
        history = torch.stack(history, dim=1).cpu().numpy() if history else None   # (B, L-1, T)
        paths = []
        for b in range(B):
            tag = best_last[b]
            path = [tag]
            for i in range(lengths[b] - 1, 0, -1):
                tag = int(history[b, i - 1, tag])
                path.append(tag)
            paths.append(path[::-1])
        return paths

In [14]:
class CharWordEncoder(nn.Module):
    # everything up to (and including) the per-tag emission scores.
    # Kept separate from the CRF so Task 3 can reuse it with a softmax on top.
    def __init__(self, n_words, n_chars, n_tags, emb_matrix,
                 char_dim=25, char_hidden=25, word_hidden=100, dropout=0.5):
        super().__init__()
        self.char_emb = nn.Embedding(n_chars, char_dim, padding_idx=0)
        self.char_lstm = nn.LSTM(char_dim, char_hidden, batch_first=True, bidirectional=True)

        self.word_emb = nn.Embedding(n_words, emb_matrix.shape[1], padding_idx=0)
        self.word_emb.weight.data.copy_(torch.from_numpy(emb_matrix))
         # GloVe is fine-tuned
        self.word_emb.weight.requires_grad = True

        self.dropout = nn.Dropout(dropout)
        self.word_lstm = nn.LSTM(emb_matrix.shape[1] + 2 * char_hidden, word_hidden,
                                 num_layers=1, batch_first=True, bidirectional=True)
        self.to_tags = nn.Linear(2 * word_hidden, n_tags)

    def char_features(self, chars):
        B, L, C = chars.shape
        flat = chars.view(B * L, C)
        lengths = (flat != 0).sum(1)
        # skip padding "words"
        real = lengths > 0
        out = torch.zeros(B * L, 2 * self.char_lstm.hidden_size, device=chars.device)

        packed = pack_padded_sequence(self.char_emb(flat[real]), lengths[real].cpu(),
                                      batch_first=True, enforce_sorted=False)
        _, (h, _) = self.char_lstm(packed)                  # h: (2, n_real_words, 25)
        out[real] = torch.cat([h[0], h[1]], dim=1)          # last forward state + last backward state
        return out.view(B, L, -1)

    def forward(self, words, chars, mask):
        x = torch.cat([self.word_emb(words), self.char_features(chars)], dim=2)
        x = self.dropout(x)                                 # dropout before the word BiLSTM, as in the paper
        lengths = mask.sum(1).cpu()
        packed = pack_padded_sequence(x, lengths, batch_first=True, enforce_sorted=False)
        out, _ = self.word_lstm(packed)
        out, _ = pad_packed_sequence(out, batch_first=True, total_length=words.size(1))
        return self.to_tags(out)                            # (B, L, n_tags) emission scores


class BiLSTMCRF(nn.Module):
    def __init__(self, encoder, n_tags):
        super().__init__()
        self.encoder = encoder
        self.crf = CRF(n_tags)

    def loss(self, words, chars, tags, mask):
        return self.crf.neg_log_likelihood(self.encoder(words, chars, mask), tags, mask)

    def predict(self, words, chars, mask):
        return self.crf.decode(self.encoder(words, chars, mask), mask)

In [15]:
encoder = CharWordEncoder(len(word_list), len(char_list), len(Labels), emb_matrix)
model = BiLSTMCRF(encoder, len(Labels)).to(device)

n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"Trainable parameters: {n_params:,}")

BiLSTMCRF(
  (encoder): CharWordEncoder(
    (char_emb): Embedding(86, 25, padding_idx=0)
    (char_lstm): LSTM(25, 25, batch_first=True, bidirectional=True)
    (word_emb): Embedding(21215, 100, padding_idx=0)
    (dropout): Dropout(p=0.5, inplace=False)
    (word_lstm): LSTM(150, 100, batch_first=True, bidirectional=True)
    (to_tags): Linear(in_features=200, out_features=9, bias=True)
  )
  (crf): CRF()
)
Trainable parameters: 2,337,558


In [16]:
BATCH_SIZE = 10
MAX_EPOCHS = 50
PATIENCE = 5
LR0, DECAY, MOMENTUM, CLIP = 0.015, 0.05, 0.9, 5.0

optimizer = torch.optim.SGD(model.parameters(), lr=LR0, momentum=MOMENTUM)

id2tag = {i: t for t, i in tagToId.items()}

def predict_split(split, batch_size=64):
    model.eval()
    preds = []
    with torch.no_grad():
        exs = encoded[split]
        for s in range(0, len(exs), batch_size):
            words, chars, _, mask = make_batch(exs[s:s + batch_size])
            for path in model.predict(words, chars, mask):
                preds.append([id2tag[t] for t in path])
    return preds

def dev_f1():
    gold = [tags for _, tags in data["validation"]]
    return f1_score(gold, predict_split("validation"))

In [17]:
import time

In [18]:
history = []
best_f1, best_epoch, bad_epochs = -1.0, -1, 0
best_state = None

for epoch in range(MAX_EPOCHS):
    lr = LR0 / (1 + DECAY * epoch)
    for g in optimizer.param_groups:
        g["lr"] = lr

    model.train()
    order = list(range(len(encoded["train"])))
    random.shuffle(order)
    total_loss, start = 0.0, time.time()

    for s in range(0, len(order), BATCH_SIZE):
        batch = [encoded["train"][i] for i in order[s:s + BATCH_SIZE]]
        words, chars, tags, mask = make_batch(batch, train=True)

        optimizer.zero_grad()
        loss = model.loss(words, chars, tags, mask)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), CLIP)
        optimizer.step()
        total_loss += loss.item() * len(batch)

    f1 = dev_f1()
    history.append({"epoch": epoch + 1, "lr": lr, "loss": total_loss / len(order), "dev_f1": f1})
    print(f"epoch {epoch + 1:2d} | lr {lr:.5f} | train loss {total_loss / len(order):7.3f} "
          f"| dev F1 {f1:.4f} | {time.time() - start:5.1f}s")

    if f1 > best_f1:
        best_f1, best_epoch, bad_epochs = f1, epoch + 1, 0
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print(f"No improvement for {PATIENCE} epochs - stopping early.")
            break

model.load_state_dict(best_state)
print(f"\nBest dev F1 = {best_f1:.4f} at epoch {best_epoch}")

epoch  1 | lr 0.01500 | train loss   2.128 | dev F1 0.8991 |  74.5s
epoch  2 | lr 0.01429 | train loss   0.951 | dev F1 0.9050 |  76.8s
epoch  3 | lr 0.01364 | train loss   0.757 | dev F1 0.9114 |  72.4s
epoch  4 | lr 0.01304 | train loss   0.633 | dev F1 0.9272 |  69.9s
epoch  5 | lr 0.01250 | train loss   0.557 | dev F1 0.9291 |  70.8s
epoch  6 | lr 0.01200 | train loss   0.494 | dev F1 0.9308 |  71.3s
epoch  7 | lr 0.01154 | train loss   0.466 | dev F1 0.9352 |  70.6s
epoch  8 | lr 0.01111 | train loss   0.413 | dev F1 0.9367 |  70.4s
epoch  9 | lr 0.01071 | train loss   0.385 | dev F1 0.9402 |  70.0s
epoch 10 | lr 0.01034 | train loss   0.362 | dev F1 0.9375 |  71.0s
epoch 11 | lr 0.01000 | train loss   0.344 | dev F1 0.9394 |  70.5s
epoch 12 | lr 0.00968 | train loss   0.323 | dev F1 0.9389 |  70.1s
epoch 13 | lr 0.00937 | train loss   0.301 | dev F1 0.9366 |  71.2s
epoch 14 | lr 0.00909 | train loss   0.287 | dev F1 0.9427 |  70.4s
epoch 15 | lr 0.00882 | train loss   0.272 | dev

In [19]:
gold = [tags for _, tags in data["test"]]
pred = predict_split("test")

p, r, f = precision_score(gold, pred), recall_score(gold, pred), f1_score(gold, pred)
flat_g = [t for s in gold for t in s]
flat_p = [t for s in pred for t in s]
acc = np.mean([a == b for a, b in zip(flat_g, flat_p)])

print("===== Test (BiLSTM-CRF) =====")
print(f"Entity-level  P = {p:.4f}  R = {r:.4f}  F1 = {f:.4f}")
print(f"Token accuracy (reference only) = {acc:.4f}")
print(classification_report(gold, pred, digits=4))

train_forms = {w for toks, _ in data["train"] for w in toks}
known, unknown = [], []
for (toks, g), q in zip(data["test"], pred):
    for w, a, b in zip(toks, g, q):
        (known if w in train_forms else unknown).append(a == b)
print(f"Known words   : {np.mean(known):.4f}  ({len(known)} tokens)")
print(f"Unknown words : {np.mean(unknown):.4f}  ({len(unknown)} tokens)")

===== Test (BiLSTM-CRF) =====
Entity-level  P = 0.9066  R = 0.9076  F1 = 0.9071
Token accuracy (reference only) = 0.9794
              precision    recall  f1-score   support

         LOC     0.9102    0.9359    0.9228      1668
        MISC     0.7865    0.7977    0.7921       702
         ORG     0.8914    0.8844    0.8879      1661
         PER     0.9728    0.9499    0.9612      1617

   micro avg     0.9066    0.9076    0.9071      5648
   macro avg     0.8902    0.8920    0.8910      5648
weighted avg     0.9072    0.9076    0.9073      5648

Known words   : 0.9859  (40779 tokens)
Unknown words : 0.9332  (5656 tokens)


In [20]:
import pandas as pd
trans = pd.DataFrame(model.crf.transitions.detach().cpu().numpy(), index=Labels, columns=Labels)
print("CRF transition scores, rows = previous tag")
display(trans.round(2))

CRF transition scores, rows = previous tag


,O,B-PER,I-PER,B-ORG,I-ORG,B-LOC,I-LOC,B-MISC,I-MISC
O,2.47,3.82,-2.91,3.03,-3.79,3.08,-3.02,1.88,-3.55
B-PER,-0.37,-2.92,6.19,-0.97,-0.79,-0.15,-0.56,-0.36,-0.55
I-PER,0.46,-1.82,3.03,-0.55,-0.67,-0.64,-0.55,-0.31,-0.42
B-ORG,0.05,-0.58,-0.85,-2.37,7.36,-0.86,-0.70,-0.22,-0.92
I-ORG,0.06,-0.52,-0.78,-1.72,5.91,-1.05,-1.06,-0.08,-0.87
B-LOC,-0.12,-0.68,-0.80,-0.33,-1.51,-2.06,5.90,0.44,-1.14
I-LOC,-0.27,-0.44,-0.47,-0.41,-1.01,-0.96,3.15,0.03,-0.57
B-MISC,-0.69,0.47,-0.82,0.68,-1.63,-0.09,-1.18,-1.71,5.82
I-MISC,-0.91,-0.46,-0.48,0.26,-1.00,-0.34,-1.11,-0.56,4.47


In [21]:
rows = []
if os.path.exists("/content/drive/MyDrive/NLP/task1_hmm.pkl"):
    with open("/content/drive/MyDrive/NLP/task1_hmm.pkl", "rb") as fh:
        t1 = pickle.load(fh)["task1_results"]
    rows.append(("Task 1 - HMM + Viterbi", t1["precision"], t1["recall"], t1["f1"]))
rows.append(("Task 2 - BiLSTM-CRF", p, r, f))

print(f"{'Model':28s} {'P':>7s} {'R':>7s} {'F1':>7s}")
for name, a, b, c in rows:
    print(f"{name:28s} {a:7.4f} {b:7.4f} {c:7.4f}")

Model                              P       R      F1
Task 1 - HMM + Viterbi        0.6885  0.6843  0.6864
Task 2 - BiLSTM-CRF           0.9066  0.9076  0.9071


In [22]:
# Everything Task 3 needs: the trained encoder, the vocabularies and the Task 2 scores
torch.save({
    "model_state": model.state_dict(),
    "encoder_config": {"n_words": len(word_list), "n_chars": len(char_list), "n_tags": len(Labels)},
    "word2id": wordToId, "char2id": charTOId, "tag2id": tagToId, "labels": Labels,
    "emb_matrix": emb_matrix,
}, "task2_bilstm_crf.pt")

with open("/content/drive/MyDrive/NLP/task2_results.json", "w") as fh:
    json.dump({"precision": p, "recall": r, "f1": f, "token_acc": float(acc),
               "best_epoch": best_epoch, "best_dev_f1": best_f1, "history": history}, fh, indent=2)

print("Saved task2_bilstm_crf.pt and task2_results.json")
print(f"TASK 2 RESULT -> test entity-level F1 = {f:.4f}")

# Optional: keep copies in Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp task2_bilstm_crf.pt task2_results.json task2_training_curve.png /content/drive/MyDrive/

Saved task2_bilstm_crf.pt and task2_results.json
TASK 2 RESULT -> test entity-level F1 = 0.9071
